In [1]:
import os, math, time, requests

PLANTA = "NunezBalboa"
ANIOS = [2016, 2019, 2022]    # antes, durante y después de la obra

# Extensión de la planta (sacada de la ortofoto 2022, con algo de margen)
LON_MIN, LON_MAX = -6.252, -6.199
LAT_MIN, LAT_MAX = 38.419, 38.456

# Teselas de 1024 px a ~0,3 m/px (unos 307 m de lado): la misma resolución
# que las imágenes con las que validé el modelo de Gemini
TAM_PX = 1024
LADO_M = 307
D_LAT = LADO_M / 111_000                                    # grados de latitud por tesela
D_LON = LADO_M / (111_320 * math.cos(math.radians(38.44)))  # grados de longitud por tesela

N_FILAS = math.ceil((LAT_MAX - LAT_MIN) / D_LAT)
N_COLS = math.ceil((LON_MAX - LON_MIN) / D_LON)

WMS = "https://www.ign.es/wms/pnoa-historico"   # servicio WMS de ortofotos históricas del IGN
BASE = f"/lakehouse/default/Files/vuelos/planta={PLANTA}"

print(f"Rejilla: {N_FILAS} x {N_COLS} = {N_FILAS * N_COLS} teselas por año")

StatementMeta(, 2e90daf4-f01b-4b8f-aca6-f6b5b90ee995, 3, Finished, Available, Finished, False)

Rejilla: 14 x 16 = 224 teselas por año


In [2]:
def descargar_tesela(anio, bbox, ruta):
    """Pide una tesela al WMS del IGN. Reintento si el servidor falla (ayer estuvo caído)."""
    params = {
        "SERVICE": "WMS", "VERSION": "1.1.1", "REQUEST": "GetMap",
        "LAYERS": f"PNOA{anio}", "STYLES": "", "SRS": "EPSG:4326",
        "BBOX": ",".join(f"{v:.6f}" for v in bbox),
        "WIDTH": TAM_PX, "HEIGHT": TAM_PX, "FORMAT": "image/jpeg",
    }
    for intento in range(3):
        try:
            r = requests.get(WMS, params=params, timeout=120)
            r.raise_for_status()
            # Si el WMS da un error, devuelve XML en vez de imagen
            if not r.headers.get("Content-Type", "").startswith("image"):
                raise ValueError(r.text[:200])
            with open(ruta, "wb") as f:
                f.write(r.content)
            return
        except Exception as e:
            print(f"  Fallo ({e.__class__.__name__}), reintento en 30 s...")
            time.sleep(30)
    raise RuntimeError(f"No se pudo descargar {ruta}")

registros = []
for anio in ANIOS:
    carpeta = f"{BASE}/fecha={anio}"
    os.makedirs(carpeta, exist_ok=True)
    nuevas = 0
    for fila in range(N_FILAS):
        for col in range(N_COLS):
            # bbox de la tesela: empiezo por la esquina noroeste
            lat_max = LAT_MAX - fila * D_LAT
            lat_min = lat_max - D_LAT
            lon_min = LON_MIN + col * D_LON
            lon_max = lon_min + D_LON
            nombre = f"tesela_f{fila:02d}_c{col:02d}.jpg"
            ruta = f"{carpeta}/{nombre}"

            # Incremental: solo descargo lo que no tengo ya
            if not os.path.exists(ruta):
                descargar_tesela(anio, (lon_min, lat_min, lon_max, lat_max), ruta)
                nuevas += 1
                time.sleep(0.2)   # no saturar el servicio del IGN

            registros.append({
                "planta": PLANTA, "anio": anio, "tesela": nombre,
                "fila": fila, "columna": col,
                "lat_min": lat_min, "lat_max": lat_max,
                "lon_min": lon_min, "lon_max": lon_max,
                "ruta": ruta.replace("/lakehouse/default/", ""),
            })
    print(f"{anio}: {nuevas} teselas nuevas descargadas")

StatementMeta(, 2e90daf4-f01b-4b8f-aca6-f6b5b90ee995, 4, Finished, Available, Finished, False)

2016: 224 teselas nuevas descargadas
2019: 224 teselas nuevas descargadas
  Fallo (HTTPError), reintento en 30 s...
  Fallo (HTTPError), reintento en 30 s...
2022: 224 teselas nuevas descargadas


In [3]:
from pyspark.sql import functions as F

# Inventario de teselas: es la tabla de entrada para el siguiente paso (Gemini)
df = (spark.createDataFrame(registros)
      .withColumn("fecha_ingesta", F.current_timestamp()))
df.write.format("delta").mode("overwrite").saveAsTable("bronze_teselas")

display(spark.sql("""
    SELECT anio, COUNT(*) AS teselas FROM bronze_teselas GROUP BY anio ORDER BY anio
"""))

StatementMeta(, 2e90daf4-f01b-4b8f-aca6-f6b5b90ee995, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ff45fafe-adb5-4451-8cfa-c1befb8fa2ad)